# Tutorial 16: Part 2 - Ball thrower

In tutorial 1, we optimized morphology and control. However, the time-dependent behavior of a system also determines possible trajectories, so in the second part we consider morphology and trajectory optimization. The task is to 

In [1]:
from thrower.parametric_arm import build, params, urdfOutPath
import numpy as np
import casadi
import pinocchio as pin
import pinocchio.casadi as cpin
import matplotlib.pyplot as plt

from robomeshcat import Robot, Scene
from IPython.display import display, IFrame

from thrower.ocp_utils import frameAcceleration, SX_zeros, framePlacementFunctions, frameVelocityFunctions, frameVelocity
import os

pybullet build time: May  8 2026 13:42:58


In [2]:
# Hyperparameters
N = 1000
T = 2.0  # final time
dt = T / N
nu = 3
nq = 3
nv = 3

In [3]:
class CoDesignProblem:

    def __init__(self, desired_position, desired_velocity):
        self.desired_position = desired_position
        self.desired_velocity = desired_velocity
        self.default_params = {
                "link_0": 1.0,
                "link_1": 1.0,
                }
        
        self.logger = {
            "params": [],
            "costs": [],
            "xs": [],
            "qs": [],
            "us": [],
            "gripperTrajectory": [],
            "gripperVelocity": []
        }
    

    def get_bounds(self):
        pass

    def fitness(self, params):
        pass

    def log(self, params, cost, xs, qs, us, gripperTrajectory, gripperVelocity):
        self.logger["params"].append(params)
        self.logger["costs"].append(cost)
        self.logger["xs"].append(xs)
        self.logger["qs"].append(qs)
        self.logger["us"].append(us)
        self.logger["gripperTrajectory"].append(gripperTrajectory)
        self.logger["gripperVelocity"].append(gripperVelocity)

    def plot_solution(self, index):
        fig, (ax0, ax1, ax2) = plt.subplots(ncols=3, constrained_layout=True)

        X = self.logger["xs"][index].T
        U = self.logger["us"][index].T

        time = np.linspace(0, T, N + 1)

        [ax0.plot(time, X[i, :]) for i in range(nq)]
        ax0.legend([f"$q_{i}$" for i in range(nq)])
        ax0.set_xlabel("t [s]")
        ax0.set_title("x")

        [ax1.plot(time, X[i, :]) for i in range(nq, nq + nv)]
        ax1.legend([f"$v_{i}$" for i in range(nv)])
        ax1.set_xlabel("t [s]")
        ax1.set_title("v")

        [ax2.plot(time[:-1], U[i, :]) for i in range(nu)]
        ax2.legend([f"$u_{i}$" for i in range(nu)])
        ax2.set_xlabel("t [s]")
        ax2.set_title("u")

        plt.show()

    def plot_control(self, index):
        fig, (ax0) = plt.subplots(ncols=1, constrained_layout=True)

        time = np.linspace(0, T, N + 1)
        U = self.logger["us"][index]

        [ax0.plot(time[:-1], U[:, i]) for i in range(nu)]
        ax0.legend([f"$u_{i}$" for i in range(nu)])
        ax0.set_xlabel("t [s]")
        ax0.set_title("u")

        plt.show()


    def plot_states(self, index, maxIndex=-1):
        fig, (ax0, ax1) = plt.subplots(ncols=2, constrained_layout=True)

        time = np.linspace(0, T, N + 1)
        time = time[:maxIndex]
        X = self.logger['xs'][index].T[:, :maxIndex]

        [ax0.plot(time, X[i, :]) for i in range(nq)]
        plt.gca().set_prop_cycle(None)
        ax0.legend([f"$q_{i}$" for i in range(nq)])
        ax0.set_xlabel("t [s]")
        ax0.set_title("x")

        [ax1.plot(time, X[i, :]) for i in range(nq, nq + nv)]
        plt.gca().set_prop_cycle(None)
        ax1.legend([f"$v_{i}$" for i in range(nv)])
        ax1.set_xlabel("t [s]")
        ax1.set_title("v")

        plt.show()
    
    def plot_gripper(self, index):
        fig, (ax0, ax1) = plt.subplots(ncols=2, constrained_layout=True)

        time = np.linspace(0, T, N + 1)
        time = time[:]
        gripper_pos = self.logger['gripperTrajectory'][index].T[:, :]
        gripper_vel = self.logger['gripperVelocity'][index].T[:, :]

        [ax0.plot(time, traj) for traj in gripper_pos]
        plt.gca().set_prop_cycle(None)
        ax0.legend(["x", "y", "z"])
        ax0.set_xlabel("t [s]")
        plt.gca().set_prop_cycle(None)
        ax0.hlines(self.desired_position, time[0], time[-1], linestyle="dashed")
        ax0.legend(["desired x", "desired y", "desired z"])
        ax0.set_title("Gripper position")

        [ax1.plot(time, traj) for traj in gripper_vel]
        ax1.legend(["v_x", "v_y", "v_z"])
        ax1.set_xlabel("t [s]")
        plt.gca().set_prop_cycle(None)
        ax1.hlines(self.desired_velocity, time[0], time[-1], linestyle="dashed")
        ax1.legend(["desired v_x", "desired v_y", "desired v_z"])
        ax1.set_title("Gripper velocity")

        plt.show()

In [4]:
def get_bounds(self):
    # Define bounds for the parameters
    return ([1.0, 1.0], [2.0, 2.0])  # Lower and upper bounds for link_0 and link_1

CoDesignProblem.get_bounds = get_bounds

In [32]:
def dynamics(self, params):
    # Build the model
    l0, l1 = params
    build({"link_0" : l0, "link_1" : l1})

    # Import the robot
    self.model = model = pin.buildModelFromUrdf(urdfOutPath)
    model.q0 = np.zeros(model.nq)
    model.effortLimit = 0.9 * model.effortLimit
    model.gravity.linear = np.array([0, 0, -9.81])
    self.velocityLimit = 30  # reaching the limits of the actuator

    """
        Preparing for the optimization
    """

    self.cmodel = cmodel = cpin.Model(model)
    self.cdata = cdata = cmodel.createData()

    cq = casadi.SX.sym("q", nq)
    cv = casadi.SX.sym("v", nv)
    ctau = casadi.SX.sym("tau", nu)
    x = casadi.vertcat(cq, cv)
    u = casadi.vertcat(ctau)

    """
        Dynamics
    """

    # Underactuation of the base: the torque is applied only to the last 2 joints
    ctau_joints = ctau

    # Unconstrained case
    # getting the joint acceleration
    a = cpin.aba(cmodel, cdata, cq, cv, ctau_joints)

    # Integrator
    v_next = cv + a * dt
    q_next = cpin.integrate(cmodel, cq, cv * dt)
    x_next = casadi.vertcat(q_next, v_next)

    # state transition function Phi(x, u) -> x+
    self.Phi = casadi.Function("Phi", [x, u], [x_next], ["x", "u"], ["x_next"])

CoDesignProblem.dynamics = dynamics 

In [34]:
def optimization_setup(self):
    """
        Optimization problem
    """

    # Casadi optimization class
    self.OCP = opti = casadi.Opti()

    # Variables MX type
    X = opti.variable(nq+nv, N + 1)   # state trajectory
    U = opti.variable(nu, N)       # control trajectory

    # Boundary conditions
    model = self.model
    cmodel = self.cmodel
    cdata = self.cdata
    data = model.createData()

    # Initial state guess - make sure the robot is standing in a non-singular position
    qJoints = np.array([0, np.pi, 0])

    gripperRotation, gripperPosition = framePlacementFunctions(
        cmodel, cdata, cq, cv, model.getFrameId("gripper")
    )
    elbowRotation, elbowPosition = framePlacementFunctions(
        cmodel, cdata, cq, cv, model.getFrameId("RY_1")
    )
    gripperVelocity, _ = frameVelocityFunctions(
        cmodel, cdata, cq, cv, model.getFrameId("gripper")
    )
    
    q0 = np.array([0, np.pi, 0])

    x0 = np.hstack((q0, np.zeros(nv)))

    # Objective function
    obj = 0
    # Lagrange term
    for i in range(N):
        obj += 1e-2 * U[:, i].T @ U[:, i] + 1e-3 * X[-nv:, i].T @ X[-nv:, i]

    # Dynamic constraints
    for k in range(int(N)):
            opti.subject_to(X[:, k + 1] == self.Phi(X[:, k], U[:, k]))

    for i in range(N):
        opti.subject_to(elbowPosition(X[:, i])[-1] >= 0.1) # pyright: ignore[reportOptionalSubscript]

    # Final state constraints and cost
    obj += 1e3 * casadi.sumsqr(X[:, 0] - x0)
    obj += 1e3 * casadi.sumsqr(X[:, -1] - x0)

    opti.subject_to(X[:, 0] == x0)
    opti.subject_to(gripperPosition(X[:, -1])[1:] == self.desired_position[1:]) # type: ignore
    opti.subject_to(gripperVelocity(X[:, -1]) == self.desired_velocity)
    # opti.subject_to(X[-nv:, -1] == 0)

    # Initial values for solver
    opti.set_initial(X, np.vstack([x0 for _ in range(N + 1)]).T)
    # opti.set_value(P, np.zeros(P.shape))

    opti.minimize(obj)

CoDesignProblem.optimization_setup = optimization_setup

In [35]:
def solve(self):
    opti = self.OCP

    # Options
    opts = {"expand" : True, "print_time":False}
    opts["ipopt"] = {"max_iter": 10000, "tol":1e-2, "print_level":0} # type: ignore

    # Initialization
    opti.solver("ipopt", opts)  # set numerical backend

    self.dvars = dvars = {'X': X, 'U': U}

    try:
        sol = opti.solve_limited()
        print("Problem converged")

    except RuntimeError as e:
        print(e)
        import warnings

        self.sol = sol = opti.debug
        warnings.simplefilter("error", UserWarning)
        print("Problem NOT converged")

    """
        Simulation of the solution in viewer
    """
    # play()

    # Retrieving the solution
    X, U = dvars['X'], dvars['U']
    us = np.array(opti.value(U.T))
    xs = np.array(opti.value(X.T))
    qs = xs[:, :nq]
    gripperTrajectory = np.array([opti.value(gripperPosition(X[:, i])) for i in range(N + 1)]) # type: ignore
    gripperVelocity = np.array([opti.value(gripperVelocity(X[:, i])) for i in range(N + 1)]) # type: ignore

    cost = np.linalg.norm(us)

    self.log(params, cost, xs, qs, us, gripperTrajectory, gripperVelocity)

    return [cost]

CoDesignProblem.solve = solve

In [36]:
def fitness(self, params):
    self.dynamics(params)
    self.optimization_setup()
    return self.solve()

CoDesignProblem.fitness = fitness

In [37]:
INITIAL_POSITION = (0, 1.0)  # Starting point is the origin

optimized_vel = [1, 0, -1]

initial_position = np.array([INITIAL_POSITION[0], 0.0, INITIAL_POSITION[1]])
problem = CoDesignProblem(desired_position = initial_position, desired_velocity = np.array(optimized_vel))

problem.fitness([1.0, 1.0])

NameError: name 'cq' is not defined

In [11]:
problem.logger

{'params': [],
 'costs': [],
 'xs': [],
 'qs': [],
 'us': [],
 'gripperTrajectory': [],
 'gripperVelocity': []}

In [ ]:
# Create the optimization problem
import pygmo as pg
prob = pg.problem(problem)

# Create a population
pop = pg.population(prob, size=5)

# Choose an algorithm (CMA-ES in this case)
algo = pg.algorithm(pg.cmaes(gen=5, force_bounds=True))

# Evolve the population
algo.set_verbosity(1)
pop = algo.evolve(pop)

# Get the best solution
best_params = pop.champion_x
best_fitness = pop.champion_f
print("Best parameters:", best_params)
print("Best fitness (negative jump height):", best_fitness)

# Visualize the best solution
params["link_0"], params["link_1"] = best_params
build(params)
viz = visualizeMeshcat(urdfOutPath)
viz.display(np.zeros(3))  # Display the robot in the neutral configuration

